# 08. V1/V2와 V3~V5 특징 병합

**목적**  
전체 성분형 V1과 선택 성분형 V2에 공통 특징 V3~V5를 결합합니다.

**입력**  
`data/interim의 V1~V5 특징 파일`

**출력**  
`data/processed/최종_V1_전체성분_V3V4V5.csv 및 최종_V2_선택성분_V3V4V5.csv`

> 저장된 특징 파일만 사용하며 외부 요청은 발생하지 않습니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


## 특징 세트 병합

V1 또는 V2 성분 특징에 V3 기능군, V4 문헌 규칙, V5 제품명 주제 특징을 결합해 두 종류의 최종 모델 입력 데이터를 생성합니다.


In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

BASE_DIR = DATA_INTERIM_DIR

V1_PATH = BASE_DIR / "V1_전체성분_원핫인코딩.csv"
V2_PATH = BASE_DIR / "V2_선택성분_원핫인코딩.csv"
V3_PATH = BASE_DIR / "V3_성분기능군_개수.csv"
V4_PATH = BASE_DIR / "V4_문헌기반_성분조합.csv"
V5_PATH = BASE_DIR / "V5_제품명_강조주제.csv"

OUTPUT_V1_PATH = DATA_PROCESSED_DIR / "최종_V1_전체성분_V3V4V5.csv"
OUTPUT_V2_PATH = DATA_PROCESSED_DIR / "최종_V2_선택성분_V3V4V5.csv"

for path in [V1_PATH, V2_PATH, V3_PATH, V4_PATH, V5_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"파일이 없습니다: {path}")

print("입력 파일 확인 완료")


In [ ]:
def safe_theme_name(theme: str) -> str:
    text = unicodedata.normalize('NFKC', str(theme)).strip()
    text = re.sub(r'[^0-9A-Za-z가-힣]+', '_', text)
    return text.strip('_')


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding='utf-8-sig')


In [ ]:
def validate_input_frames(v1, v2, v3, v4, v5):
    required = {
        'v1': {
            'product_id',
            'product_name_raw',
            'product_name_clean',
        },
        'v2': {
            'product_id',
            'product_name',
            '피부타입',
            'formula_group',
        },
        'v3': {
            'product_id',
            'product_name',
            'v3_total_ingredient_count',
        },
        'v4': {
            'product_id',
            'product_name',
            '피부타입',
            'formula_group',
            'skin_건성',
            'skin_민감성',
            'skin_복합성',
            'skin_약건성',
            'skin_중성',
            'skin_지성',
            'skin_트러블성',
        },
        'v5': {
            'product_id',
            'product_name_raw',
            'product_name_clean',
            '강조문구_1',
            '강조문구_2',
            '강조문구_3',
        },
    }

    frames = {
        'v1': v1,
        'v2': v2,
        'v3': v3,
        'v4': v4,
        'v5': v5,
    }

    for name, frame in frames.items():
        missing = required[name] - set(frame.columns)

        if missing:
            raise ValueError(
                f'{name} 필수 컬럼 누락: {sorted(missing)}'
            )

        frame['product_id'] = (
            frame['product_id']
            .astype(str)
            .str.strip()
        )

    for name in ['v1', 'v3', 'v5']:
        if frames[name]['product_id'].duplicated().any():
            raise ValueError(
                f'{name}: product_id 중복이 있습니다.'
            )

    for name in ['v2', 'v4']:
        if frames[name][
            ['product_id', '피부타입']
        ].duplicated().any():
            raise ValueError(
                f'{name}: product_id+피부타입 중복이 있습니다.'
            )

    product_sets = {
        name: set(frame['product_id'])
        for name, frame in frames.items()
    }

    reference = product_sets['v1']

    for name, product_set in product_sets.items():
        if product_set != reference:
            raise ValueError(
                f'{name} 제품 집합이 V1과 다릅니다.'
            )

    v2_keys = set(
        map(
            tuple,
            v2[['product_id', '피부타입']]
            .to_numpy(),
        )
    )
    v4_keys = set(
        map(
            tuple,
            v4[['product_id', '피부타입']]
            .to_numpy(),
        )
    )

    if v2_keys != v4_keys:
        raise ValueError(
            'V2와 V4의 product_id+피부타입 키가 다릅니다.'
        )


In [ ]:
def prepare_v1(v1):
    id_columns = {
        'product_id',
        'product_name_raw',
        'product_name_clean',
    }

    ingredient_columns = [
        column
        for column in v1.columns
        if column not in id_columns
    ]

    rename_map = {}

    for column in ingredient_columns:
        normalized = unicodedata.normalize(
            'NFKC',
            str(column),
        ).strip()

        rename_map[column] = (
            normalized
            if normalized.startswith('v1_')
            else f'v1_{normalized}'
        )

    prepared = (
        v1[['product_id'] + ingredient_columns]
        .rename(columns=rename_map)
    )

    v1_columns = [
        rename_map[column]
        for column in ingredient_columns
    ]

    if not prepared.columns.is_unique:
        raise ValueError(
            'V1 성분 컬럼명이 중복됩니다.'
        )

    unique_values = set(
        pd.unique(
            prepared[
                v1_columns
            ].to_numpy().ravel()
        )
    )

    if not unique_values.issubset({0, 1}):
        raise ValueError(
            f'V1 원핫값 오류: {sorted(unique_values)[:10]}'
        )

    return prepared, v1_columns


In [ ]:
def prepare_v2(v2):
    v2_columns = [
        column
        for column in v2.columns
        if column.startswith('v2_')
    ]

    prepared = v2[
        ['product_id', '피부타입']
        + v2_columns
    ].copy()

    unique_values = set(
        pd.unique(
            prepared[
                v2_columns
            ].to_numpy().ravel()
        )
    )

    if not unique_values.issubset({0, 1}):
        raise ValueError(
            f'V2 원핫값 오류: {sorted(unique_values)[:10]}'
        )

    return prepared, v2_columns


def prepare_v3(v3):
    v3_columns = [
        'v3_total_ingredient_count'
    ] + [
        column
        for column in v3.columns
        if column.startswith('v3_count_')
    ]

    v3_columns = list(
        dict.fromkeys(v3_columns)
    )

    if len(v3_columns) != 13:
        raise ValueError(
            'V3는 total 1개 + count 12개여야 합니다. '
            f'현재 {len(v3_columns)}개'
        )

    return (
        v3[['product_id'] + v3_columns].copy(),
        v3_columns,
    )


In [ ]:
def prepare_v5(v5):
    source_columns = [
        '강조문구_1',
        '강조문구_2',
        '강조문구_3',
    ]

    themes = sorted({
        str(theme).strip()
        for column in source_columns
        for theme in v5[column].dropna().unique()
        if str(theme).strip()
    })

    prepared = v5[['product_id']].copy()
    v5_columns = []

    for theme in themes:
        column = (
            f'v5_theme_{safe_theme_name(theme)}'
        )

        prepared[column] = (
            v5[source_columns]
            .eq(theme)
            .any(axis=1)
            .astype('int8')
        )

        v5_columns.append(column)

    return prepared, v5_columns, themes


## 실행 및 최종 CSV 저장


In [ ]:
v1 = read_csv(V1_PATH)
v2 = read_csv(V2_PATH)
v3 = read_csv(V3_PATH)
v4 = read_csv(V4_PATH)
v5 = read_csv(V5_PATH)


In [ ]:
validate_input_frames(
    v1,
    v2,
    v3,
    v4,
    v5,
)

v1_prepared, v1_columns = prepare_v1(v1)
v2_prepared, v2_columns = prepare_v2(v2)
v3_prepared, v3_columns = prepare_v3(v3)
v5_prepared, v5_columns, themes = prepare_v5(v5)

management_columns = [
    'product_id',
    'product_name',
    '피부타입',
    'formula_group',
]

skin_columns = [
    'skin_건성',
    'skin_민감성',
    'skin_복합성',
    'skin_약건성',
    'skin_중성',
    'skin_지성',
    'skin_트러블성',
]

all_v4_columns = [
    column
    for column in v4.columns
    if column.startswith('v4_')
]

redundant_v4_map = {
    'v4_has_niacinamide': (
        'v1_나이아신아마이드',
        'v2_나이아신아마이드',
    ),
    'v4_has_panthenol': (
        'v1_판테놀',
        'v2_판테놀',
    ),
    'v4_has_ceramide': (
        'v1_세라마이드엔피',
        'v2_세라마이드엔피',
    ),
    'v4_has_cholesterol': (
        'v1_콜레스테롤',
        'v2_콜레스테롤',
    ),
}

base = v4[
    management_columns + skin_columns
].copy()


In [ ]:
check_v1 = (
    base[
        ['product_id', '피부타입']
    ]
    .merge(
        v1_prepared,
        on='product_id',
        how='left',
        validate='many_to_one',
    )
    .merge(
        v4[
            ['product_id', '피부타입']
            + list(redundant_v4_map)
        ],
        on=['product_id', '피부타입'],
        how='left',
        validate='one_to_one',
    )
)

check_v2 = (
    base[
        ['product_id', '피부타입']
    ]
    .merge(
        v2_prepared,
        on=['product_id', '피부타입'],
        how='left',
        validate='one_to_one',
    )
    .merge(
        v4[
            ['product_id', '피부타입']
            + list(redundant_v4_map)
        ],
        on=['product_id', '피부타입'],
        how='left',
        validate='one_to_one',
    )
)

redundant_v4_columns = []

for (
    v4_column,
    (v1_column, v2_column),
) in redundant_v4_map.items():

    v1_equal = (
        v1_column in check_v1.columns
        and check_v1[v4_column]
        .equals(check_v1[v1_column])
    )

    v2_equal = (
        v2_column in check_v2.columns
        and check_v2[v4_column]
        .equals(check_v2[v2_column])
    )

    if v1_equal and v2_equal:
        redundant_v4_columns.append(
            v4_column
        )
    else:
        print(
            f'[유지] {v4_column}: '
            '성분 원핫과 완전히 같지 않음'
        )


In [ ]:
v4_columns = [
    column
    for column in all_v4_columns
    if column not in redundant_v4_columns
]

v4_prepared = v4[
    ['product_id', '피부타입']
    + v4_columns
].copy()

final_v1 = (
    base
    .merge(
        v1_prepared,
        on='product_id',
        how='inner',
        validate='many_to_one',
    )
    .merge(
        v3_prepared,
        on='product_id',
        how='inner',
        validate='many_to_one',
    )
    .merge(
        v4_prepared,
        on=['product_id', '피부타입'],
        how='inner',
        validate='one_to_one',
    )
    .merge(
        v5_prepared,
        on='product_id',
        how='inner',
        validate='many_to_one',
    )
)

final_v2 = (
    base
    .merge(
        v2_prepared,
        on=['product_id', '피부타입'],
        how='inner',
        validate='one_to_one',
    )
    .merge(
        v3_prepared,
        on='product_id',
        how='inner',
        validate='many_to_one',
    )
    .merge(
        v4_prepared,
        on=['product_id', '피부타입'],
        how='inner',
        validate='one_to_one',
    )
    .merge(
        v5_prepared,
        on='product_id',
        how='inner',
        validate='many_to_one',
    )
)


In [ ]:
final_v1 = final_v1[
    management_columns
    + skin_columns
    + v1_columns
    + v3_columns
    + v4_columns
    + v5_columns
]

final_v2 = final_v2[
    management_columns
    + skin_columns
    + v2_columns
    + v3_columns
    + v4_columns
    + v5_columns
]

expected_rows = len(base)
key_columns = [
    'product_id',
    '피부타입',
]

for name, frame in [
    ('final_v1', final_v1),
    ('final_v2', final_v2),
]:
    if len(frame) != expected_rows:
        raise ValueError(
            f'{name} 행 수 오류'
        )

    if frame[
        key_columns
    ].duplicated().any():
        raise ValueError(
            f'{name} 키 중복'
        )

    if frame.isna().sum().sum() != 0:
        raise ValueError(
            f'{name} 결측값 존재'
        )

    if not frame.columns.is_unique:
        raise ValueError(
            f'{name} 컬럼명 중복'
        )

    forbidden = [
        column
        for column in frame.columns
        if (
            column.startswith('v3_ratio_')
            or column in redundant_v4_columns
        )
    ]

    if forbidden:
        raise ValueError(
            f'{name} 금지 컬럼 포함: {forbidden}'
        )


In [ ]:
common_columns = (
    management_columns
    + skin_columns
    + v3_columns
    + v4_columns
    + v5_columns
)

if not final_v1[
    common_columns
].equals(
    final_v2[common_columns]
):
    raise ValueError(
        'V1/V2의 공통 V3·V4·V5 블록이 다릅니다.'
    )

info = {
    'management_columns': management_columns,
    'skin_columns': skin_columns,
    'v1_columns': v1_columns,
    'v2_columns': v2_columns,
    'v3_columns': v3_columns,
    'v4_columns': v4_columns,
    'v5_columns': v5_columns,
    'themes': themes,
    'removed_v4_columns': redundant_v4_columns,
}


In [ ]:
final_v1.to_csv(
    OUTPUT_V1_PATH,
    index=False,
    encoding='utf-8-sig',
)

final_v2.to_csv(
    OUTPUT_V2_PATH,
    index=False,
    encoding='utf-8-sig',
)

print('저장 완료')
print('1.', OUTPUT_V1_PATH)
print('2.', OUTPUT_V2_PATH)

print('\n최종 크기')
print('- V1+V345:', final_v1.shape)
print('- V2+V345:', final_v2.shape)

print('\n블록별 피처 수')
print('- 피부유형:', len(info['skin_columns']))
print('- V1 전성분:', len(info['v1_columns']))
print('- V2 주성분:', len(info['v2_columns']))
print('- V3 Count:', len(info['v3_columns']))
print('- V4 조합:', len(info['v4_columns']))
print('- V5 테마:', len(info['v5_columns']))

print(
    '- 제거한 V4 중복:',
    info['removed_v4_columns'],
)
print('- V5 테마:', info['themes'])


## 모델 입력 컬럼

`product_id`, `product_name`, `피부타입`, `formula_group`은 식별·분할 컬럼이며, 나머지 V1~V5 컬럼을 모델 입력 특징으로 사용합니다.
